# 02. 데이터 전처리

`src/data/preprocessor.py`를 사용해 raw 데이터를 정제하고 `data/processed/`에 저장합니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from src.data.preprocessor import DialoguePreprocessor

# 한글 폰트 설정 (NanumGothic)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('경로 설정 완료 / 폰트:', plt.rcParams['font.family'])

## 1. 원본 데이터 로드

In [2]:
train_raw = pd.read_csv(RAW_DIR / 'train.csv')
dev_raw   = pd.read_csv(RAW_DIR / 'dev.csv')
test_raw  = pd.read_csv(RAW_DIR / 'test.csv')

print(f'train: {len(train_raw):,}  |  dev: {len(dev_raw):,}  |  test: {len(test_raw):,}')
train_raw.head(2)

train: 12,457  |  dev: 499  |  test: 499


,fname,dialogue,summary,topic
0,train_0,"#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나...","스미스씨가 건강검진을 받고 있고, 호킨스 의사는 매년 건강검진을 받는 것을 권장합니...",건강검진 받기
1,train_1,"#Person1#: 안녕하세요, 파커 부인, 어떻게 지내셨나요?\n#Person2#...",파커 부인이 리키를 데리고 백신 접종을 하러 갔다. 피터스 박사는 기록을 확인한 후...,백신


## 2. 전처리 실행

### 전처리 옵션
| 옵션 | 설명 | 기본값 |
|---|---|---|
| `prompt_template` | 모델 입력 포맷 (`{dialogue}` 포함) | 대화문 그대로 |
| `anonymize_speakers` | `#PersonN#` → `화자N` 변환 | `False` |
| `max_dialogue_len` | 이 길이 초과 샘플 제거 | `None` (제거 없음) |

In [ ]:
# SOLAR-10.7B-Instruct 학습용 프롬프트 (System / User / Assistant 포맷)
PROMPT_TEMPLATE = """### System:
당신은 한국어 대화 요약 전문가입니다. 대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. 요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. 핵심 내용만 1~3문장으로 간결하게 요약하세요. 반드시 한국어로 작성하되, CPU·USB 등 고유 영문 약어나 전문 용어는 그대로 사용해도 됩니다.

### User:
아래 대화를 읽고 핵심 내용을 한국어로 요약해주세요. 화자 태그(#Person1# 등)를 유지하세요.

{dialogue}

### Assistant:
{summary}"""

preprocessor = DialoguePreprocessor(
    prompt_template=PROMPT_TEMPLATE,
    anonymize_speakers=False,
    max_dialogue_len=None,
)

train = preprocessor.process(train_raw)
dev   = preprocessor.process(dev_raw)
test  = preprocessor.process(test_raw)

print(f'전처리 완료 → train: {len(train):,}  |  dev: {len(dev):,}  |  test: {len(test):,}')

## 3. 전처리 결과 확인

In [4]:
idx = 0
print('=== 원본 dialogue ===')
print(repr(train_raw['dialogue'].iloc[idx][:200]))
print()
print('=== 전처리 후 dialogue ===')
print(repr(train['dialogue'].iloc[idx][:200]))
print()
print('=== input_text (프롬프트 포함) ===')
print(train['input_text'].iloc[idx][:400])

=== 원본 dialogue ===
'#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?\n#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.\n#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.\n#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?\n#Person1#'

=== 전처리 후 dialogue ===
'#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?\n#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.\n#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.\n#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?\n#Person1#'

=== input_text (프롬프트 포함) ===
다음 대화를 한국어로 요약하세요.

[대화]
#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?
#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.
#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.
#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?
#Person1#: 심각한 질병을 피하는 가장 좋은 방법은 이를 조기에 발견하는 것입니다. 그러니 당신의 건강을 위해 최소한 매년 한 번은 오세요.
#Person2#: 알겠습니다.
#Person1#: 여기 보세요. 당신의 눈과 귀는 괜찮아 보입니다. 깊게 숨을 들이쉬세요. 스미스씨, 담배 피우시나요?
#Person2#: 네.
#


In [5]:
# summary 전처리 전/후 비교
# \n 포함된 샘플 확인
newline_mask = train_raw['summary'].str.contains('\n', na=False)
print(f'원본 summary \\n 포함: {newline_mask.sum()}개')
print(f'전처리 후 \\n 포함  : {train["summary"].str.contains(chr(10)).sum()}개')

원본 summary \n 포함: 12개
전처리 후 \n 포함  : 0개


In [6]:
# 전처리 후 길이 통계
stats = pd.DataFrame({
    'dialogue_len_mean':  [train['dialogue'].str.len().mean(), dev['dialogue'].str.len().mean()],
    'dialogue_len_max':   [train['dialogue'].str.len().max(),  dev['dialogue'].str.len().max()],
    'input_text_len_mean':[train['input_text'].str.len().mean(), dev['input_text'].str.len().mean()],
    'summary_len_mean':   [train['summary'].str.len().mean(), dev['summary'].str.len().mean()],
}, index=['train', 'dev']).round(1)
stats

,dialogue_len_mean,dialogue_len_max,input_text_len_mean,summary_len_mean
train,438.7,2546,470.7,87.4
dev,432.5,1484,464.5,81.7


## 4. 저장

In [7]:
train.to_csv(PROC_DIR / 'train.csv', index=False)
dev.to_csv(  PROC_DIR / 'dev.csv',   index=False)
test.to_csv( PROC_DIR / 'test.csv',  index=False)

print('저장 완료')
for f in PROC_DIR.glob('*.csv'):
    print(f'  {f.name}: {pd.read_csv(f).shape}')

저장 완료
  test.csv: (499, 3)
  dev.csv: (499, 5)
  train.csv: (12457, 5)


## 5. DataLoader 동작 확인

In [8]:
from transformers import AutoTokenizer
from src.data.dataset import DialogueSummarizationDataset
from torch.utils.data import DataLoader

MODEL_NAME = 'gogamza/kobart-base-v2'   # 테스트용 경량 모델
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = DialogueSummarizationDataset(
    train,
    tokenizer=tokenizer,
    max_input_len=512,
    max_target_len=128,
    model_type='seq2seq',
)

loader = DataLoader(train_ds, batch_size=4, shuffle=False)
batch  = next(iter(loader))

print('batch keys:', list(batch.keys()))
for k, v in batch.items():
    print(f'  {k}: {v.shape}  dtype={v.dtype}')

config.json: 0.00B [00:00, ?B/s]

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


batch keys: ['input_ids', 'attention_mask', 'labels']
  input_ids: torch.Size([4, 512])  dtype=torch.int64
  attention_mask: torch.Size([4, 512])  dtype=torch.int64
  labels: torch.Size([4, 128])  dtype=torch.int64


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:3856: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
